# Site4Drug End-to-End Reproducible Demo\n\nThis notebook runs the full demo pipeline end-to-end:\n1. Resolve input sequence\n2. Run constraint-first site discovery\n3. Inspect ranked candidates + evidence + report\n4. Write a reproducibility manifest with config + artifact hashes + Top-K signature\n

## 0) Prerequisites\n\n```bash\ncd <repo-root>\nsource .venv/bin/activate\npython -m pip install -r requirements.txt\n```\n

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "README.md").exists() and (candidate / "pipeline").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory.")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)


In [ ]:
from getpass import getpass

from site4drug_inference.demo.notebook_utils import ensure_api_key_or_raise

try:
    ensure_api_key_or_raise(REPO_ROOT)
    print("TINKER_API_KEY loaded.")
except RuntimeError:
    key = getpass("Paste TINKER_API_KEY (input hidden): ").strip()
    if not key:
        raise RuntimeError("No key provided.")
    os.environ["TINKER_API_KEY"] = key
    ensure_api_key_or_raise(REPO_ROOT)
    print("TINKER_API_KEY set for this notebook session.")


In [ ]:
import importlib
from site4drug_inference.common.model_defaults import BASE_MODEL as _BASE_MODEL
from site4drug_inference.demo import notebook_utils as nb_utils

nb_utils = importlib.reload(nb_utils)
DEFAULT_BASE_MODEL = getattr(nb_utils, "DEFAULT_BASE_MODEL", _BASE_MODEL)
DEFAULT_CHECKPOINT = nb_utils.DEFAULT_CHECKPOINT
print("DEFAULT_CHECKPOINT =", DEFAULT_CHECKPOINT)
resolve_input_sequence = nb_utils.resolve_input_sequence
run_notebook_prediction = nb_utils.run_notebook_prediction
set_reproducible_session = nb_utils.set_reproducible_session

REPRO_SEED = 7
SESSION_INFO = set_reproducible_session(REPRO_SEED)
print("Session info:", SESSION_INFO)

CONFIG = {
    "uniprot": "P29996",
    "mode": "auto",
    "candidate_source": "llm_propose",
    "top_k": 5,
    "use_base_model": False,
    "base_model": DEFAULT_BASE_MODEL,
    "checkpoint": DEFAULT_CHECKPOINT,
    "max_tokens": 2048,
    "max_input_tokens": 10000,
    "output_dir": str(REPO_ROOT / "outputs" / "predictions"),
    "use_multi_agent": True,
    "repair_with_base_model": True,
    "ptm_source": "multi_rule",
    "ptm_policy": "tiered",
    "motif_source": "remote",
    "use_motif": True,
    "use_iedb_validation": True,
    "iedb_table_path": str(REPO_ROOT / "data/tcell_regions_with_seq.parquet"),
    "iedb_iou_threshold": 0.3,
    "sequence_file": "examples/antigens/P29996.fasta",
    "sequence_text": "",
    "allow_online_lookup": True,
}

raw_sequence, input_source = resolve_input_sequence(
    uniprot=CONFIG["uniprot"],
    sequence_text=CONFIG["sequence_text"],
    sequence_file=CONFIG["sequence_file"],
    allow_online_lookup=CONFIG["allow_online_lookup"],
)
print("Input source:", input_source)
print("Sequence length:", len(raw_sequence))


In [ ]:
result = run_notebook_prediction(
    uniprot=CONFIG["uniprot"],
    raw_sequence=raw_sequence,
    checkpoint=CONFIG["checkpoint"],
    base_model=CONFIG["base_model"],
    use_base_model=CONFIG["use_base_model"],
    mode=CONFIG["mode"],
    candidate_source=CONFIG["candidate_source"],
    top_k=CONFIG["top_k"],
    max_tokens=CONFIG["max_tokens"],
    max_input_tokens=CONFIG["max_input_tokens"],
    output_dir=CONFIG["output_dir"],
    enable_plot=True,
    use_multi_agent=CONFIG["use_multi_agent"],
    input_source=input_source,
    ptm_source=CONFIG["ptm_source"],
    ptm_policy=CONFIG["ptm_policy"],
    motif_source=CONFIG["motif_source"],
    use_motif=CONFIG["use_motif"],
    use_iedb_validation=CONFIG["use_iedb_validation"],
    iedb_table_path=CONFIG["iedb_table_path"],
    iedb_iou_threshold=CONFIG["iedb_iou_threshold"],
)

run_payload = result["run_payload"]
run_dir = result["run_dir"]
print("Run dir:", run_dir)
print("Recommended modality:", run_payload["recommended_modality"])
print("Modality confidence:", run_payload["modality_confidence"])
print("Token strategy:", run_payload["token_strategy_used"])
print("IEDB validation:", run_payload.get("iedb_validation", {}))


In [ ]:
from site4drug_inference.demo.notebook_utils import (
    build_repro_manifest,
    save_repro_manifest,
    save_run_snapshot,
    topk_signature,
    validate_demo_run_payload,
)

schema_check = validate_demo_run_payload(run_payload)
print("Schema check passed:", schema_check["ok"])
if not schema_check["ok"]:
    print("Schema issues:", schema_check["issues"])

manifest = build_repro_manifest(
    result=result,
    config=CONFIG,
    session_info=SESSION_INFO,
    notebook_path="notebooks/demo/Site4Drug_end_to_end_repro.ipynb",
)
manifest_path = save_repro_manifest(manifest, run_dir / "repro_manifest.json")
snapshot_path = run_dir / "prediction_log_snapshot.json"
save_run_snapshot(run_payload, snapshot_path)

print("Manifest:", manifest_path)
print("Snapshot:", snapshot_path)
print("Top-K signature:")
for row in topk_signature(run_payload):
    print(" ", row)


In [ ]:
import pandas as pd
from IPython.display import HTML, Image, display

from site4drug_inference.demo.notebook_utils import ranked_candidates_df, candidate_evidence_df

rank_df = ranked_candidates_df(run_payload)
evidence_df = candidate_evidence_df(run_payload)

display(rank_df)
display(evidence_df.head(20))

plot_name = run_payload.get("plot_artifacts", {}).get("plot_png_name")
if plot_name:
    display(Image(filename=str(run_dir / plot_name)))
else:
    print("No plot artifact generated.")

display(HTML((run_dir / "prediction_report.html").read_text(encoding="utf-8")))

manifest_df = pd.DataFrame(manifest.get("artifacts", []))
display(manifest_df)


In [ ]:
RUN_REPLAY_CHECK = False

if RUN_REPLAY_CHECK:
    replay_result = run_notebook_prediction(
        uniprot=CONFIG["uniprot"],
        raw_sequence=raw_sequence,
        checkpoint=CONFIG["checkpoint"],
        base_model=CONFIG["base_model"],
        use_base_model=CONFIG["use_base_model"],
        mode=CONFIG["mode"],
        candidate_source=CONFIG["candidate_source"],
        top_k=CONFIG["top_k"],
        max_tokens=CONFIG["max_tokens"],
        max_input_tokens=CONFIG["max_input_tokens"],
        output_dir=CONFIG["output_dir"],
        enable_plot=True,
        use_multi_agent=CONFIG["use_multi_agent"],
        input_source=f"{input_source}:replay",
        repair_with_base_model=CONFIG["repair_with_base_model"],
        ptm_source=CONFIG["ptm_source"],
        ptm_policy=CONFIG["ptm_policy"],
        motif_source=CONFIG["motif_source"],
        use_motif=CONFIG["use_motif"],
        use_iedb_validation=CONFIG["use_iedb_validation"],
        iedb_table_path=CONFIG["iedb_table_path"],
        iedb_iou_threshold=CONFIG["iedb_iou_threshold"],
    )
    sig_a = topk_signature(run_payload)
    sig_b = topk_signature(replay_result["run_payload"])
    print("Replay run dir:", replay_result["run_dir"])
    print("Top-K signature identical:", sig_a == sig_b)
    if sig_a != sig_b:
        print("Primary signature:")
        for row in sig_a:
            print(" ", row)
        print("Replay signature:")
        for row in sig_b:
            print(" ", row)
else:
    print("Replay check skipped. Set RUN_REPLAY_CHECK=True to compare two runs with the same config.")
